# Neuro-Symbolic Pipeline v12.1 — Z3-informed Fallback + Qwen-priority Arbiter
**EXACT 2026 — XAI Challenge @ IJCNN | Qwen3-8B + Z3 + vLLM | Kaggle T4×2**

## Kết quả v12 (baseline cho v12.1): 52.34% (425/812)

Quan trọng: **Z3 bucket 60.0%** chạm chính xác trần lý thuyết 60.9% từ thí nghiệm sạch 46 câu Yes/No FOL gold → entailment-thuần không vắt thêm được điểm. Hai chỗ rò rỉ rõ ràng:

| Tier | Đúng/Tổng | Acc | Bản chất |
|------|-----------|-----|----------|
| z3_arbiter_agree | 8/9 | **88.9%** | Tin tưởng được, giữ |
| z3_trust + z3_direct | 321/536 | 59.9% | Chạm trần ngữ nghĩa "No"-class |
| **z3_arbiter_override** | **3/8** | **37.5%** | **BUG logic — flip** |
| qwen_fallback | 93/258 | 36.0% | Prompt thiếu thông tin formal |

## Hai thay đổi v12 → v12.1

### Thay đổi 1: Flip arbiter override (5 dòng)
v12 logic: Z3 conf ≥ 0.8 ∧ Qwen bất đồng → ghi đè Qwen bằng Z3 → **sai 5/8 lần**.

Diễn giải: cụm "Z3 conf cao + Qwen không đồng ý" chính là chữ ký của khoảng cách ngữ nghĩa "No"-class — Z3 over-claim entailment trên statement dạng converse/inverse, Qwen thì hiểu ngữ cảnh pedagogical. **Khi bất đồng, luôn ưu tiên Qwen.** Z3 chỉ thắng khi đồng thuận.

### Thay đổi 2: Z3-informed Qwen fallback prompt (đòn bẩy chính)
v12 prompt fallback chỉ đưa premises-NL → Qwen đoán mò → 36%. v12.1 prompt include:
- Premise NL **+ premise FOL** song song
- Câu hỏi
- **Z3 evidence**: vote distribution + verdict tạm + confidence
- Cảnh báo về over-claim ở statement converse/free-var

Qwen lúc này không đoán mò mà *reason informed*: thấy Z3 nói Yes với conf 0.6 nhưng statement là converse của P3 → quyết định No. Kỳ vọng qwen_fallback: 36% → 45-50%.

## Cấu hình mới
```python
USE_Z3_INFORMED_FALLBACK = False   # bật mặc định v12.1
```

## Dự kiến v12.1
- Flip override: +5/812 = +0.6pp
- Z3-informed prompt (đẩy qwen_fallback 36% → 45%): +23/812 = +2.8pp
- Tổng kỳ vọng: **55-57%**

## Roadmap sau v12.1
1. Nếu accuracy ≥ 55% → train **v13 calibrator** (LightGBM trên train split) để phá trần 60% Z3-bucket
2. Nếu accuracy < 54% → BoN diversity issue, thử BoN_TEMP=0.5 hoặc TAU=0.7

## Phương pháp v12.1 (sơ đồ)
```
premises-FOL ──► FOLParser ──► Z3 exprs (gold, 99% SAT)
                                    │
For each question:
  ├─ Câu có sẵn FOL ──► parse trực tiếp (z3_direct)
  └─ Còn lại ──► Pass-2 BoN=5 → ground → entail → vote
                                                    │
                              ┌─────────────────────┴───────────┐
                              │  T1 z3_trust:  conf≥0.6, definite, grounded≥0.6
                              │  T2 z3_arbiter (cần Qwen kiểm chứng):
                              │       agree     → z3_arbiter_agree
                              │       disagree  → qwen_arbiter (v12.1: KHÔNG override)
                              │  T3 fallback: Qwen với prompt Z3-informed
                              └─────────────────────────────────┘
```

## Lưu ý cho người chạy
File output: `/kaggle/working/pipeline_results_v12_1.json` (đã đổi từ v12).
Runtime kỳ vọng: ~14-15 phút (tăng nhẹ do prompt Qwen dài hơn).


In [2]:
#!/usr/bin/env python3
"""
notebook_v12_inference.py -- Neuro-Symbolic Pipeline v12
EXACT 2026 -- XAI Challenge @ IJCNN | Qwen3-8B + Z3 + vLLM | Kaggle T4x2

v12 = v11 + FOL-string Pass-2 + BoN Self-Consistency + Predicate Grounding
        + 3-tier Confidence Gating + idx Filter + Unsat-Core Explanation

See markdown cell above for changes and method.
"""


'\nnotebook_v12_inference.py -- Neuro-Symbolic Pipeline v12\nEXACT 2026 -- XAI Challenge @ IJCNN | Qwen3-8B + Z3 + vLLM | Kaggle T4x2\n\nv12 = v11 + FOL-string Pass-2 + BoN Self-Consistency + Predicate Grounding\n        + 3-tier Confidence Gating + idx Filter + Unsat-Core Explanation\n\nSee markdown cell above for changes and method.\n'

In [3]:
# Fix Kaggle T4
import os
os.environ['VLLM_USE_V1'] = '0'


In [4]:

# ==================================================================
# STAGE 0 -- Install Dependencies & Config
# ==================================================================

import subprocess, sys

pkgs = [
    "vllm",
    "z3-solver",
]
print("Installing vLLM + z3-solver (co the mat 2-5 phut)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages"]
    + pkgs,
    check=True,
)
print("All packages installed OK")

import json, os, re, time, csv, traceback
from pathlib import Path
from dataclasses import dataclass, field


# --- v8.1 FIX: KILL FLASHINFER AFTER PIP INSTALL ---
import os, sys, shutil, glob
for d in glob.glob("/usr/local/lib/python3.12/dist-packages/flashinfer*"):
    tgt = d + "_DISABLED"
    if os.path.exists(d) and not os.path.exists(tgt):
        try: os.rename(d, tgt)
        except: shutil.rmtree(d, ignore_errors=True)
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
for mod in list(sys.modules.keys()):
    if "flashinfer" in mod: del sys.modules[mod]
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASH_ATTN"
# ---------------------------------------------------

import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from z3 import (
    Int, IntSort, IntVal, BoolSort, Function,
    ForAll, Exists, And, Or, Not, Implies, Solver, sat, unsat,
)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}    : {props.name}  ({props.total_memory / 1024**3:.1f} GB)")
print("Imports OK")


Installing vLLM + z3-solver (co the mat 2-5 phut)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.9/360.9 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 90.2 MB/s eta 0:00

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cud

All packages installed OK
PyTorch  : 2.11.0+cu130
CUDA OK  : True
GPU 0    : Tesla T4  (14.6 GB)
GPU 1    : Tesla T4  (14.6 GB)
Imports OK


In [5]:

# ==================================================================
# CAU HINH -- Chinh sua o day
# ==================================================================
QWEN_MODEL_ID  = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
DATASET_PATH   = "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries (2).json"
PHYSICS_DATASET_PATH = ""  # v8.2: Skip physics, focus on logic

# --- Data Split Config ---
SEED = 42
TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
RUN_ON_SPLIT = "all"  # v8.2: Full dataset  # 'test', 'train', 'val', hoac 'all'

N_SAMPLES      = 411
# --- Best-of-N Config ---
BEST_OF_N       = 3  # v8.2: Reduced for time budget     # Number of candidates per sample
BON_TEMPERATURE = 0.5  # v8.2: Lower for consistency   # Higher temp for diversity

# --- v9: Optional CoT for Formalization (Pass 1) ---
# False = giu nguyen baseline v8.2 (da kiem chung 50.0%).
# True  = cho LLM "thinking" truoc khi chot JSON AST -> ky vong giam no_ast, danh thuc Z3.
FORMALIZE_WITH_COT = False

MAX_RETRIES    = 2               # Giam xuong 2 vi v4 nhe hon
OUTPUT_PATH    = "/kaggle/working/pipeline_results_v12_1.json"
MAX_NEW_TOKENS_PASS1 = 4096      # Token cho Premises
MAX_NEW_TOKENS_PASS2 = 1024      # Token cho Question (it hon)
ANS_MAX_TOKENS       = 600       # Token cho Qwen Fallback

# vLLM Config
TENSOR_PARALLEL  = min(N_GPUS, 2)
MAX_MODEL_LEN    = 8192
GPU_MEMORY_UTIL  = 0.85
DTYPE            = "half"

# Physics Config
PHYSICS_MODE       = "direct"
PHYSICS_MAX_TOKENS = 1024
PHYSICS_TOLERANCE  = 0.05

# Z3 Entailment Config
Z3_ENTAILMENT      = True
Z3_SOLVER_TIMEOUT  = 5000
Z3_SELF_CORRECT    = False   # v10: OFF (24/114 resolved, 9 -> Unknown; low ROI)
Z3_SC_MAX_RETRIES  = 1       # Max self-correction rounds
REPETITION_PENALTY = 1.1     # v8.1 from exact.ipynb: chong token loop
# ==================================================================

# ==================================================================
# v10 -- Z3 FIDELITY CONFIG
# ==================================================================
# Quality Gate: chi cho Z3 tra loi khi formalization dang tin cay.
Z3_QUALITY_GATE          = True
GATE_REQUIRE_FULL_COMPILE = True   # compiled_count == total_count
GATE_REJECT_HALLUCINATION = True   # 0 hallucination warning
# Neu gate fail -> cau hoi do chuyen sang Qwen CoT fallback.

# Formalizer-LoRA (STaR loop). De rong "" = dung model goc cho moi pass.
FORMALIZER_LORA_PATH = ""          # vd: "/kaggle/working/qwen3_formalizer_lora"
LORA_MAX_RANK        = 16

# Export verified formalizations (chay tren TRAIN de tao data train cho LoRA)
EXPORT_VERIFIED        = False
VERIFIED_OUTPUT_PATH   = "/kaggle/working/verified_formalizations.json"

# ==================================================================
# v11 -- GOLD-FOL CONFIG
# ==================================================================
PREMISE_SOURCE   = "gold"     # "gold" (parse premises-FOL) | "lora_fol" (NL->FOL via LoRA)
USE_IDX_FILTER   = False      # True: chi dung premise trong idx[q] (gold supporting set)
FOL_FALLBACK_TO_QWEN = True   # mau parse FOL that bai -> Qwen CoT truc tiep

# ==================================================================
# v12 -- BoN + SELF-CONSISTENCY + GATING + UNSAT-CORE
# ==================================================================
BON_N_QFORMALIZE     = 5           # candidates per Pass-2 question
BON_TEMP             = 0.7         # generation diversity
SC_CONFIDENCE_TAU    = 0.6         # Tier-1 vote-share threshold
GROUNDED_FRAC_TAU    = 0.6         # Tier-1 grounded-candidates threshold
EXTRACT_UNSAT_CORE   = True        # produce support_idx for Z3=Yes

# v12.1 -- post-mortem fixes
USE_Z3_INFORMED_FALLBACK = True   # Qwen fallback sees premise FOL + Z3 votes + Z3 verdict
                                  # (v12 showed qwen_fallback @ 36%; this aims for 45-50%)
MAX_PASS2_TOKENS_YN  = 200
MAX_PASS2_TOKENS_MC  = 500

# v12 overrides v11 defaults:
USE_IDX_FILTER       = True        # ON by default in v12 (was False in v11)

print("Config OK - v12.1 (Z3-informed fallback + Qwen-priority arbiter)")


Config OK - v12.1 (Z3-informed fallback + Qwen-priority arbiter)


In [6]:

# ==================================================================
# STAGE 1 -- Load vLLM Engine
# ==================================================================

print(f"\nLoading vLLM engine ({QWEN_MODEL_ID})...")
print("  (Lan dau tai ~15 GB tu HuggingFace, sau do cache)")

t_load = time.time()

_USE_LORA = bool(FORMALIZER_LORA_PATH)
llm = LLM(
    model=QWEN_MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    dtype=DTYPE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    trust_remote_code=True,
    enforce_eager=True,        # v8.1 FIX: Disable CUDA graph to save 1.67 GiB VRAM on T4
    enable_lora=_USE_LORA,     # v10: serve formalizer LoRA if provided
    max_lora_rank=LORA_MAX_RANK,
)

# v10: build LoRA request (used ONLY for formalization passes)
LORA_REQ = None
if _USE_LORA:
    from vllm.lora.request import LoRARequest
    LORA_REQ = LoRARequest("formalizer", 1, FORMALIZER_LORA_PATH)
    print(f"Formalizer-LoRA enabled: {FORMALIZER_LORA_PATH}")
else:
    print("No formalizer LoRA (base model for all passes)")

tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_ID, trust_remote_code=True
)

t_loaded = time.time() - t_load
print(f"vLLM Engine loaded in {t_loaded:.1f}s")
print("OK")




Loading vLLM engine (/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1)...
  (Lan dau tai ~15 GB tu HuggingFace, sau do cache)
INFO 05-31 13:13:02 [utils.py:278] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'max_model_len': 8192, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': '/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1'}
WARNING 05-31 13:13:02 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_USE_V1
WARNING 05-31 13:13:02 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_ATTENTION_BACKEND
INFO 05-31 13:13:22 [model.py:617] Resolved architecture: Qwen3ForCausalLM
WARNING 05-31 13:13:22 [model.py:2090] Casting torch.bfloat16 to torch.float16.
INFO 05-31 13:13:22 [model.py:1752] Using max model len 8192
INFO 05-31 13:13:22 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-31 13:13:22 [vllm.py:977] Asynchronous scheduling

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(Worker_TP0 pid=248) INFO 05-31 13:14:09 [weight_utils.py:856] Prefetching checkpoint files: 10% (1/3)


Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:18<01:12, 18.00s/it]


(Worker_TP0 pid=248) INFO 05-31 13:14:19 [weight_utils.py:856] Prefetching checkpoint files: 20% (2/3)
(Worker_TP1 pid=249) INFO 05-31 13:14:20 [weight_utils.py:856] Prefetching checkpoint files: 10% (1/2)


Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:23<00:31, 10.47s/it]


(Worker_TP1 pid=249) INFO 05-31 13:14:24 [weight_utils.py:856] Prefetching checkpoint files: 20% (2/2)
(Worker_TP1 pid=249) INFO 05-31 13:14:24 [weight_utils.py:879] Prefetching checkpoint files into page cache finished in 23.34s
(Worker_TP0 pid=248) INFO 05-31 13:14:24 [weight_utils.py:856] Prefetching checkpoint files: 30% (3/3)
(Worker_TP0 pid=248) INFO 05-31 13:14:24 [weight_utils.py:879] Prefetching checkpoint files into page cache finished in 23.82s


Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:26<00:14,  7.18s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:29<00:05,  5.38s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:30<00:00,  3.86s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:30<00:00,  6.05s/it]
(Worker_TP0 pid=248) 


(Worker_TP0 pid=248) INFO 05-31 13:14:31 [default_loader.py:397] Loading weights took 30.31 seconds
(Worker_TP0 pid=248) INFO 05-31 13:14:32 [model_runner.py:295] Model loading took 7.64 GiB and 31.943157 seconds
(Worker_TP1 pid=249) INFO 05-31 13:14:32 [model_runner.py:295] Model loading took 7.64 GiB and 32.077179 seconds
(Worker_TP0 pid=248) INFO 05-31 13:14:43 [gpu_worker.py:466] Available KV cache memory: 4.11 GiB
(EngineCore pid=222) INFO 05-31 13:14:43 [kv_cache_utils.py:1733] GPU KV cache size: 59,856 tokens
(EngineCore pid=222) INFO 05-31 13:14:43 [kv_cache_utils.py:1734] Maximum concurrency for 8,192 tokens per request: 7.31x
(Worker_TP1 pid=249) INFO 05-31 13:15:06 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(Worker_TP0 pid=248) INFO 05-31 13:15:06 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=222) INFO 05-31 13

In [7]:

# ==================================================================
# STAGE 2 -- Ontology & Prompts (Two-Pass)
# ==================================================================

GLOBAL_ONTOLOGY_TEXT = """
## GLOBAL ONTOLOGY -- BAT BUOC TUAN THU

### Quantifiers:
  forall -> forall  |  exists -> exists

### Logical Operators:
  and, or, implies, iff, not

### AST Node Types (4 loai):
  quantifier : { "type":"quantifier",  "operator":"forall|exists",
                 "bound_variables":["x",...], "body":{...} }
  connective : { "type":"connective",  "operator":"and|or|implies|iff|not",
                 "operands":[{...},...] }
  predicate  : { "type":"predicate",   "name":"PredicateName",
                 "arguments":["x","y",...] }
  variable   : { "type":"variable",    "name":"x" }
  constant   : { "type":"constant",    "name":"SomeName" }

### QUY TAC:
  1. Chi dung 4 node types tren
  2. 'not' chi co DUNG 1 operand
  3. 'implies' co DUNG 2 operands
  4. bound_variables phai la list
  5. Variables: lowercase (x,y,z), Constants: PascalCase
"""

# ------------------------------------------------------------------
# FEW-SHOT EXAMPLES (CRITICAL FOR QWEN-7B)
# ------------------------------------------------------------------

FEW_SHOT_PREMISES = """
### VÍ DỤ HOÀN CHỈNH:

Premises:
  "All students who study hard pass the exam."
  "Alice is a student."
  "Alice studies hard."

Output:
{
  "step1_local_ontology": [
    {"predicate": "Student",   "arity": 1, "description": "x is a student"},
    {"predicate": "StudyHard", "arity": 1, "description": "x studies hard"},
    {"predicate": "PassExam",  "arity": 1, "description": "x passes the exam"}
  ],
  "step2_premises_ast": [
    {
      "premise_id": 0,
      "source_nl": "All students who study hard pass the exam.",
      "ast": {
        "type": "quantifier", "operator": "forall",
        "bound_variables": ["x"],
        "body": {
          "type": "connective", "operator": "implies",
          "operands": [
            {"type": "connective", "operator": "and", "operands": [
              {"type": "predicate", "name": "Student",   "arguments": ["x"]},
              {"type": "predicate", "name": "StudyHard", "arguments": ["x"]}
            ]},
            {"type": "predicate", "name": "PassExam", "arguments": ["x"]}
          ]
        }
      }
    },
    { "premise_id": 1, "source_nl": "Alice is a student.",
      "ast": {"type": "predicate", "name": "Student", "arguments": ["Alice"]} },
    { "premise_id": 2, "source_nl": "Alice studies hard.",
      "ast": {"type": "predicate", "name": "StudyHard", "arguments": ["Alice"]} }
  ]
}
"""

FEW_SHOT_QUESTIONS = """
### VÍ DỤ HOÀN CHỈNH:
Question 1: "Is it true that Alice passes the exam?" (Yes/No)
Question 2: "Which is correct? A. Alice fails. B. Alice passes." (Multiple Choice)

Output:
{
  "step3_question_fol": [
    {
      "question_id": 0,
      "question_type": "yes_no",
      "statement_ast": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
    },
    {
      "question_id": 1,
      "question_type": "multiple_choice",
      "options_ast": {
         "A": {"type": "connective", "operator": "not", "operands": [{"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}]},
         "B": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
      }
    }
  ]
}
"""

# ------------------------------------------------------------------
# PASS 1 PROMPTS (PREMISES ONLY)
# ------------------------------------------------------------------

PREMISE_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Buoc 1: Tao LOCAL ONTOLOGY -- danh sach Predicates\n"
    "  Buoc 2: Chuyen TUNG tien de thanh cay AST JSON de quy\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_PREMISES + "\n"
    "Output JSON THUAN TUY (khong markdown, khong text thua):\n"
    '{\n'
    '  "step1_local_ontology": [\n'
    '    {"predicate": "Name", "arity": N, "description": "..."}\n'
    '  ],\n'
    '  "step2_premises_ast": [\n'
    '    {"premise_id": 0, "source_nl": "...", "ast": { <AST> }}\n'
    '  ]\n'
    '}\n'
)

CORRECTION_SYSTEM = (
    "Ban la chuyen gia sua loi FOL. He thong Z3 da phat hien loi.\n"
    "Nhiem vu: sua lai Buoc 1 + Buoc 2 de khong con loi.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    "Output JSON thuan tuy -- format GIONG HET lan dau.\n"
)

# ------------------------------------------------------------------
# PASS 2 PROMPTS (QUESTIONS ONLY)
# ------------------------------------------------------------------

QUESTION_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Chuyen TUNG cau hoi thanh AST JSON de kiem tra Z3 Entailment.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_QUESTIONS + "\n"
    "QUAN TRONG:\n"
    "  - Ban PHAI su dung cac Predicate tu LOCAL ONTOLOGY duoc cung cap.\n"
    "  - question_type: 'yes_no' hoac 'multiple_choice'\n"
    "  - yes_no: chuyen statement thanh 1 AST node (statement_ast)\n"
    "  - multiple_choice: chuyen TUNG option A/B/C/D thanh AST (options_ast)\n\n"
    "Output JSON THUAN TUY:\n"
    '{\n'
    '  "step3_question_fol": [\n'
    '    {\n'
    '      "question_id": 0,\n'
    '      "question_type": "yes_no",\n'
    '      "source_nl": "statement to check",\n'
    '      "statement_ast": { <AST> }\n'
    '    }\n'
    '  ]\n'
    '}\n'
)

# Fallback
# Fallback -- exact.ipynb CoT format
ANSWER_SYSTEM = (
    "You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).\n"
    "Given a set of premises (in natural language and FOL notation), you must:\n"
    "1. Analyze the logical structure of each premise carefully.\n"
    "2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation.\n"
    "3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.\n"
    "4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.\n"
    "5. For Yes/No questions: verify the statement logically.\n"
    "6. If the premises are INSUFFICIENT, your Final Answer MUST be exactly: Unknown\n"
    "7. Never hallucinate a conclusion not entailed by the premises.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your reasoning here>\n"
    "### Final Answer\n"
    "<A or B or C or D or Yes or No or Unknown>"
)

# Physics
PHYSICS_SOLVER_SYSTEM = (
    "You are an expert physics problem solver.\n"
    "Solve the problem step-by-step, showing all calculations.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your detailed solution here>\n"
    "### Final Answer\n"
    "<your numerical answer with unit>"
)
PHYSICS_MC_SYSTEM = (
    "You are an expert physics problem solver.\n"
    "Solve the multiple-choice problem step-by-step.\n"
    "Evaluate each option against the physics principles.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your detailed solution here>\n"
    "### Final Answer\n"
    "<A or B or C or D or your numerical answer>"
)

# ==================================================================
# UTILS
# ==================================================================
import os, csv, re, time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

def load_dataset(path, is_physics=False, max_samples=N_SAMPLES, split_mode=RUN_ON_SPLIT):
    if not path or not os.path.exists(path): return []
    if path.endswith(".csv"):
        with open(path, encoding="utf-8") as f: raw = list(csv.DictReader(f))
    else:
        with open(path, encoding="utf-8") as f: raw = json.load(f)
    
    out = raw[:max_samples]
    
    # --- SPLIT LOGIC ---
    import random
    rng = random.Random(SEED)
    rng.shuffle(out)
    n = len(out)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    
    if split_mode == "train":
        out = out[:n_train]
    elif split_mode == "val":
        out = out[n_train:n_train+n_val]
    elif split_mode == "test":
        out = out[n_train+n_val:]
    # if split_mode == "all", keep out as is

    if is_physics and out:
        for s in out:
            s["premises-NL"] = []
            s["questions"]   = [s.get("question", "")]
            s["answers"]     = [str(s.get("answer", "Unknown"))]
            s["_unit"]       = s.get("unit", "")
            s["_is_physics"] = True
    return out

def safe_json(text):
    text = text.strip()
    # v7 FIX: Strip Qwen3 <think>...</think> tags
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    try: return json.loads(text)
    except: pass
    m = re.search(r"```(?:json)?\s*\n?(.*?)```", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1).strip())
        except: pass
    start = text.find("{")
    if start >= 0:
        depth, end = 0, start
        for i in range(start, len(text)):
            if text[i] == "{": depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break
        try: return json.loads(text[start : end + 1])
        except: pass
    return {}

def batch_generate(prompt_pairs, max_tokens, enable_thinking=True, use_lora=False):
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": usr_msg}]
        try:
            formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking))
        except TypeError:
            formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    params = SamplingParams(temperature=0.05, max_tokens=max_tokens, repetition_penalty=1.1)
    _lr = LORA_REQ if (use_lora and LORA_REQ is not None) else None
    outputs = llm.generate(formatted, params, lora_request=_lr) if _lr else llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [o.outputs[0].text.strip() for o in outputs_sorted]

def batch_generate_bon(prompt_pairs, max_tokens, n=BEST_OF_N, enable_thinking=True, use_lora=False):
    """Generate N candidates per prompt using higher temperature."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            formatted.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking))
        except TypeError:
            formatted.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True))

    params = SamplingParams(
        temperature=BON_TEMPERATURE,
        max_tokens=max_tokens,
        repetition_penalty=1.1,
        n=n,
    )
    _lr = LORA_REQ if (use_lora and LORA_REQ is not None) else None
    outputs = llm.generate(formatted, params, lora_request=_lr) if _lr else llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))

    return [
        [o.text.strip() for o in out.outputs]
        for out in outputs_sorted
    ]


def z3_select_best(candidates):
    """
    v8.1: Returns (data, status, func_cache, const_map).
    """
    PRIORITY = {"sat": 4, "unsat": 3, "unknown": 2, "compile_error": 1, "no_ast": 0}
    best_data, best_status, best_score = {}, "no_ast", -1
    best_func_cache = {}
    best_const_map = {}

    for raw in candidates:
        data = safe_json(raw)
        premises_ast = data.get("step2_premises_ast", [])

        if not premises_ast:
            score = PRIORITY["no_ast"]
            status = "no_ast"
            candidate_cache = {}
            candidate_const = {}
        else:
            candidate_cache = {}
            candidate_const = {}
            z3_info = verify_with_z3(premises_ast, func_cache=candidate_cache, const_map=candidate_const)
            status = z3_info["status"]
            score = PRIORITY.get(status, 0)

        if score > best_score:
            best_score = score
            best_status = status
            best_data = data
            best_func_cache = candidate_cache
            best_const_map = candidate_const

        if best_score >= 4:  # sat -> stop early
            break

    return best_data, best_status, best_func_cache, best_const_map




import re

def extract_final_answer(model_output):
    """
    v8.1: Robust answer extraction from exact.ipynb
    Multi-pattern fallback to minimize UNPARSEABLE results.
    Priority: Final Answer block > inline patterns > standalone letters > keyword scan
    """
    text = model_output.strip()

    # Pattern 1: "### Final Answer" block
    match = re.search(
        r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)",
        text, re.IGNORECASE
    )
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        if re.match(r"^unknown", ans, re.IGNORECASE):
            return "Unknown"
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        if re.match(r"^yes", ans, re.IGNORECASE):
            return "Yes"
        if re.match(r"^no\b", ans, re.IGNORECASE):
            return "No"

    # Pattern 2: "answer is X" or "Answer: X"
    match2 = re.search(
        r"(?:answer\s*(?:is|:)\s*)([A-D]|unknown|yes|no)",
        text, re.IGNORECASE
    )
    if match2:
        g = match2.group(1).strip()
        if g.lower() == "unknown": return "Unknown"
        if g.lower() == "yes":    return "Yes"
        if g.lower() == "no":     return "No"
        return g.upper()

    # Pattern 3: Standalone letter near end of text
    last_500 = text[-500:]
    match3 = re.findall(r"\b([A-D])\b", last_500)
    if match3:
        return match3[-1].upper()

    # Pattern 4: Unknown/Yes/No anywhere
    if re.search(r"\bunknown\b", text, re.IGNORECASE):
        return "Unknown"
    if re.search(r"\byes\b", text, re.IGNORECASE):
        return "Yes"
    if re.search(r"\bno\b", text, re.IGNORECASE):
        return "No"

    return "UNPARSEABLE"

print("extract_final_answer v8.1 (from exact.ipynb) san sang")
def extract_physics_answer(model_output):
    """Extract answer from physics CoT response (v8.2)."""
    text = model_output.strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Pattern 1: ### Final Answer block
    match = re.search(r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)", text, re.IGNORECASE)
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        ans = re.sub(r"^(?:the\s+answer\s+is|answer\s*[:=])\s*", "", ans, flags=re.IGNORECASE).strip()
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        return ans
    # Pattern 2: JSON fallback
    data = safe_json(text)
    if data.get("answer"):
        return str(data["answer"]).strip()
    # Pattern 3: "answer is X"
    match2 = re.search(r"answer\s*(?:is|:|=)\s*(.+?)(?:\n|$)", text, re.IGNORECASE)
    if match2:
        return match2.group(1).strip()
    return "Unknown"

print("extract_physics_answer v8.2 san sang")



extract_final_answer v8.1 (from exact.ipynb) san sang
extract_physics_answer v8.2 san sang


In [8]:

# ==================================================================
# STAGE 3 -- Z3 Compiler + Entailment Checker (v8.1 HARDENED)
# ==================================================================
# v8.1 over v7:
#   1. const_map: Deterministic constant->IntVal mapping (no hash collision)
#   2. SAFE fuzzy: case-insensitive + prefix-strip ONLY (no substring!)
#   3. MC elimination: if 3/4 options contradicted, pick the survivor
# v8.1 over v8:
#   - REMOVED dangerous substring matching
#   - Self-Correction will NOT override Qwen fallback with Unknown
# ==================================================================


def get_z3_func(name, arity, func_cache):
    """Get or create Z3 Function with SAFE fuzzy matching.
    v8.1: Only case-insensitive + prefix-strip. NO substring matching."""
    key = f"{name}_{arity}"
    if key in func_cache:
        return func_cache[key]

    # --- v8.1: SAFE fuzzy match (no substring!) ---
    def _normalize(n):
        """Strip Is/Has/Can prefix for matching."""
        n_low = n.lower()
        # v10 FIX: removed can/can_ -- modal verbs (ability/permission) are NOT
        # the same predicate as their bare form (actuality). Caused wrong entailments.
        for prefix in ("is_", "has_", "is", "has"):
            if n_low.startswith(prefix) and len(n_low) > len(prefix):
                return n_low[len(prefix):]
        return n_low

    for cached_key in list(func_cache.keys()):
        parts = cached_key.rsplit("_", 1)
        if len(parts) != 2:
            continue
        cached_name, cached_arity_str = parts
        try:
            cached_arity = int(cached_arity_str)
        except ValueError:
            continue
        if cached_arity != arity:
            continue

        # Level 1: Case-insensitive exact match
        if cached_name.lower() == name.lower():
            func_cache[key] = func_cache[cached_key]
            print(f"    [FUZZY] {name}/{arity} -> {cached_name} (case)")
            return func_cache[key]

        # Level 2: Prefix-strip match (IsStudent -> student == student)
        if _normalize(name) == _normalize(cached_name):
            func_cache[key] = func_cache[cached_key]
            print(f"    [FUZZY] {name}/{arity} -> {cached_name} (prefix-strip)")
            return func_cache[key]

        # v8.1: NO Level 3 substring matching -- it caused false positives!

    # No fuzzy match -- create new function
    sorts = [IntSort()] * arity + [BoolSort()]
    func_cache[key] = Function(name, *sorts)
    return func_cache[key]


def _resolve_bound_var_name(bv):
    if isinstance(bv, dict):
        return bv.get("name", str(bv))
    return str(bv)


def _get_constant_val(name, const_map):
    """v8: Deterministic constant mapping. No hash collisions."""
    if name not in const_map:
        const_map[name] = IntVal(len(const_map) + 1)
    return const_map[name]


def _resolve_predicate_arg(a, var_map, func_cache, const_map):
    if isinstance(a, str):
        if a in var_map:
            return var_map[a]
        return _get_constant_val(a, const_map)
    if isinstance(a, dict):
        atype = a.get("type", "")
        name = a.get("name", "")
        if atype == "variable":
            if name in var_map:
                return var_map[name]
            v = Int(name)
            var_map[name] = v
            return v
        if atype == "constant":
            return _get_constant_val(name, const_map)
        raise ValueError(f"Argument khong hop le (type={atype!r})")
    return _get_constant_val(str(a), const_map)


def compile_ast(node, var_map, func_cache, const_map):
    """Compile 1 AST node -> Z3 expression.
    v8.1: Uses shared func_cache + const_map."""
    if not isinstance(node, dict):
        raise ValueError(f"Expected dict, got {type(node)}: {node!r}")

    ntype = node.get("type", "")

    if ntype == "quantifier":
        op = node.get("operator", "").lower()
        bvs = node.get("bound_variables", [])
        if not bvs:
            raise ValueError("quantifier thieu bound_variables")
        bv_names = [_resolve_bound_var_name(bv) for bv in bvs]
        z3_bvs = [Int(v) for v in bv_names]
        child_map = {**var_map, **{v: z3_bvs[i] for i, v in enumerate(bv_names)}}
        body = compile_ast(node["body"], child_map, func_cache, const_map)
        if op == "forall":
            return ForAll(z3_bvs, body)
        elif op in ("exists", "exist"):
            return Exists(z3_bvs, body)
        else:
            raise ValueError(f"Quantifier khong hop le: {op!r}")

    elif ntype == "connective":
        op = node.get("operator", "").lower()
        ops = [compile_ast(o, var_map, func_cache, const_map) for o in node.get("operands", [])]
        if op == "and":
            return And(*ops)
        elif op == "or":
            return Or(*ops)
        elif op == "implies":
            if len(ops) != 2:
                raise ValueError(f"implies can 2 operands, nhan {len(ops)}")
            return Implies(ops[0], ops[1])
        elif op == "iff":
            if len(ops) != 2:
                raise ValueError(f"iff can 2 operands, nhan {len(ops)}")
            return And(Implies(ops[0], ops[1]), Implies(ops[1], ops[0]))
        elif op == "not":
            if len(ops) != 1:
                raise ValueError(f"not can 1 operand, nhan {len(ops)}")
            return Not(ops[0])
        else:
            raise ValueError(f"Connective khong hop le: {op!r}")

    elif ntype == "predicate":
        name = node.get("name", "")
        args = node.get("arguments", [])
        if not name:
            raise ValueError('predicate thieu "name"')
        func = get_z3_func(name, len(args), func_cache)
        z3_args = [_resolve_predicate_arg(a, var_map, func_cache, const_map) for a in args]
        return func(*z3_args)

    elif ntype in ("variable", "constant"):
        name = node.get("name", "")
        if name in var_map:
            return var_map[name]
        if ntype == "constant":
            return _get_constant_val(name, const_map)
        v = Int(name)
        var_map[name] = v
        return v

    else:
        raise ValueError(f"AST node type khong hop le: {ntype!r}")


def verify_with_z3(premises_ast, func_cache=None, const_map=None):
    """Compile all premises -> Z3, check consistency."""
    if func_cache is None:
        func_cache = {}
    if const_map is None:
        const_map = {}
    solver = Solver()
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast = item.get("ast", {})
            if not ast:
                errors.append(f"Premise {pid}: AST rong")
                continue
            expr = compile_ast(ast, {}, func_cache, const_map)
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"Premise {pid}: {str(e)[:250]}")

    if errors:
        return {"status": "compile_error", "errors": errors,
                "compiled_count": compiled, "total_count": len(premises_ast)}
    try:
        result = solver.check()
        return {"status": str(result), "errors": [],
                "compiled_count": compiled, "total_count": len(premises_ast)}
    except Exception as e:
        return {"status": "solver_error", "errors": [str(e)],
                "compiled_count": compiled, "total_count": len(premises_ast)}


def hallucination_check(local_ontology, premises_ast):
    """Check: all predicates in AST must be in Local Ontology."""
    declared = set()
    for item in local_ontology:
        if isinstance(item, dict):
            declared.add(item.get("predicate", ""))

    warnings = []

    def collect_predicates(node, found):
        if not isinstance(node, dict):
            return
        if node.get("type") == "predicate":
            found.add(node.get("name", ""))
        for v in node.values():
            if isinstance(v, dict):
                collect_predicates(v, found)
            elif isinstance(v, list):
                for sub in v:
                    collect_predicates(sub, found)

    for item in premises_ast:
        used = set()
        collect_predicates(item.get("ast", {}), used)
        hallucinated = used - declared - {""}
        if hallucinated:
            warnings.append(
                f"Premise {item.get('premise_id', '?')}: "
                f"predicates not in Ontology -> {hallucinated}"
            )
    return warnings


# ==================================================================
# Z3 ENTAILMENT CHECKER (v8.1 HARDENED)
# ==================================================================

def _compile_premises_to_solver_shared(premises_ast, func_cache, const_map):
    """Compile premises using SHARED func_cache + const_map."""
    solver = Solver()
    solver.set("timeout", Z3_SOLVER_TIMEOUT)
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast_node = item.get("ast", {})
            if not ast_node:
                continue
            expr = compile_ast(ast_node, {}, func_cache, const_map)
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"P{pid}: {str(e)[:200]}")

    return solver, compiled, errors


def z3_entailment_check(premises_ast, question_fol_item, func_cache=None, const_map=None):
    """v8.1: Shared func_cache + const_map + safe fuzzy matching."""
    if func_cache is None:
        func_cache = {}
    if const_map is None:
        const_map = {}

    q_type = question_fol_item.get("question_type", "yes_no")

    if q_type == "yes_no":
        return _entail_yes_no(premises_ast, question_fol_item, func_cache, const_map)
    elif q_type == "multiple_choice":
        return _entail_mc(premises_ast, question_fol_item, func_cache, const_map)
    else:
        return {"answer": "Unknown", "method": "unsupported_qtype"}


def _entail_yes_no(premises_ast, q_item, func_cache, const_map):
    """Entailment for Yes/No questions."""
    stmt_ast = q_item.get("statement_ast", {})
    if not stmt_ast:
        return {"answer": "Unknown", "method": "no_statement_ast"}

    try:
        stmt_expr = compile_ast(stmt_ast, {}, func_cache, const_map)
    except Exception as e:
        return {"answer": "Unknown", "method": "stmt_compile_error",
                "detail": str(e)[:200]}

    # Test YES: premises + NOT(stmt) -> UNSAT?
    solver1, c1, e1 = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
    if e1:
        return {"answer": "Unknown", "method": "premise_compile_error",
                "detail": "; ".join(e1[:3])}

    try:
        solver1.push()
        solver1.add(Not(stmt_expr))
        r1 = solver1.check()
        solver1.pop()
    except Exception as e:
        r1 = None

    if r1 == unsat:
        return {"answer": "Yes", "method": "z3_entailment",
                "detail": "premises + NOT(Q) is UNSAT => Q is entailed"}

    # Test NO: premises + stmt -> UNSAT?
    solver2, _, _ = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
    try:
        solver2.push()
        solver2.add(stmt_expr)
        r2 = solver2.check()
        solver2.pop()
    except Exception as e:
        r2 = None

    if r2 == unsat:
        return {"answer": "No", "method": "z3_negation",
                "detail": "premises + Q is UNSAT => Q contradicts premises"}

    return {"answer": "Unknown", "method": "z3_undetermined",
            "detail": f"Neither entailed nor contradicted (r1={r1}, r2={r2})"}


def _entail_mc(premises_ast, q_item, func_cache, const_map):
    """Entailment for Multiple Choice (A/B/C/D) with elimination logic."""
    options_ast = q_item.get("options_ast", {})
    if not options_ast:
        return {"answer": "Unknown", "method": "no_options_ast"}

    entailed = []
    consistent = []
    contradicted = []
    details = {}

    for label in ("A", "B", "C", "D"):
        opt_ast = options_ast.get(label, {})
        if not opt_ast:
            continue

        try:
            opt_expr = compile_ast(opt_ast, {}, func_cache, const_map)
        except Exception as e:
            details[label] = f"compile_error: {str(e)[:100]}"
            continue

        solver, c, e = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
        if e:
            details[label] = "premise_error"
            continue

        try:
            # Entailment: premises + NOT(option) -> UNSAT?
            solver.push()
            solver.add(Not(opt_expr))
            r = solver.check()
            solver.pop()

            if r == unsat:
                entailed.append(label)
                details[label] = "entailed"

            # Contradiction: premises + option -> UNSAT?
            solver.push()
            solver.add(opt_expr)
            r2 = solver.check()
            solver.pop()

            if r2 == unsat:
                contradicted.append(label)
                if label not in details:
                    details[label] = "contradicted"
            elif r2 == sat:
                consistent.append(label)
                if label not in details:
                    details[label] = "consistent"
        except Exception as e:
            details[label] = f"solver_error: {str(e)[:100]}"

    # Decision logic (v8 enhanced)
    if len(entailed) == 1:
        return {"answer": entailed[0], "method": "z3_unique_entailment",
                "detail": f"Only {entailed[0]} entailed. {details}"}
    elif len(entailed) > 1:
        return {"answer": entailed[0], "method": "z3_multi_entailment",
                "detail": f"Multiple entailed: {entailed}. {details}"}

    # v8: If only 1 option is NOT contradicted, pick it
    non_contradicted = [l for l in ("A", "B", "C", "D")
                        if l in consistent and l not in contradicted]
    if len(non_contradicted) == 1:
        return {"answer": non_contradicted[0], "method": "z3_elimination",
                "detail": f"Elimination: only {non_contradicted[0]} survives. {details}"}

    if len(consistent) == 1:
        return {"answer": consistent[0], "method": "z3_unique_consistent",
                "detail": f"Only {consistent[0]} consistent. {details}"}
    else:
        return {"answer": "Unknown", "method": "z3_mc_undetermined",
                "detail": f"Entailed={entailed}, Consistent={consistent}, "
                          f"Contradicted={contradicted}. {details}"}


print("Z3 compiler v8.1 (const_map + SAFE fuzzy + elimination) san sang")


Z3 compiler v8.1 (const_map + SAFE fuzzy + elimination) san sang


In [9]:
# ==================================================================
# STAGE 2.5 -- Deterministic FOL-string -> Z3 Parser (v11)
# 99.6% formula / 98.8% sample coverage on the challenge dataset.
# ==================================================================
import re
from z3 import (Int, IntSort, BoolSort, Function, ForAll, Exists, IntVal,
                And, Or, Not, Implies, Solver, sat, unsat)

TOKEN_RE = re.compile(r"∀|∃|→|↔|¬|∧|∨|≥|≤|≠|>=|<=|!=|=|>|<|\+|\-|\*|/|\(|\)|,|'[^']*'|\d+\.?\d*|[A-Za-z_][A-Za-z0-9_]*|\S")
CMP = {'=','>','<','≥','≤','≠','>=','<=','!='}
def tokenize(s):
    return TOKEN_RE.findall(s)

class FOLParser:
    """v11 parser: FOL subset + arithmetic comparisons -> Z3.
    Bool functions (predicates) and Int functions (numeric terms) kept in
    separate caches keyed by name_arity."""
    def __init__(self, func_cache, const_map, int_cache=None):
        self.func_cache = func_cache      # Bool-valued predicates
        self.const_map = const_map
        self.int_cache = int_cache if int_cache is not None else {}  # Int-valued funcs

    def parse(self, s):
        self.toks = tokenize(s); self.pos = 0
        self.scope = {}; self.free = {}
        expr = self._iff()
        if self.pos != len(self.toks):
            raise ValueError(f"trailing tokens: {self.toks[self.pos:]}")
        if self.free:
            expr = ForAll(list(self.free.values()), expr)
        return expr

    def _peek(self): return self.toks[self.pos] if self.pos < len(self.toks) else None
    def _eat(self, t=None):
        cur = self._peek()
        if cur is None: raise ValueError("unexpected end")
        if t is not None and cur != t: raise ValueError(f"expected {t}, got {cur}")
        self.pos += 1; return cur

    def _iff(self):
        left = self._implies()
        while self._peek() == '↔':
            self._eat(); right = self._implies()
            left = And(Implies(left, right), Implies(right, left))
        return left
    def _implies(self):
        left = self._or()
        if self._peek() == '→':
            self._eat(); return Implies(left, self._implies())
        return left
    def _or(self):
        left = self._and()
        while self._peek() == '∨':
            self._eat(); left = Or(left, self._and())
        return left
    def _and(self):
        left = self._not()
        while self._peek() == '∧':
            self._eat(); left = And(left, self._not())
        return left
    def _not(self):
        if self._peek() == '¬':
            self._eat(); return Not(self._not())
        return self._quant_or_atom()
    def _quant_or_atom(self):
        t = self._peek()
        if t in ('∀','∃'):
            self._eat(); var = self._eat()
            v = Int(var); saved = self.scope.get(var); self.scope[var] = v
            body = self._not()
            if saved is None: del self.scope[var]
            else: self.scope[var] = saved
            return ForAll([v], body) if t=='∀' else Exists([v], body)
        if t == '(':
            self._eat('('); e = self._iff(); self._eat(')'); return e
        return self._atom()
    def _atom(self):
        # Could be a Bool predicate, or an arithmetic comparison.
        # Try to parse an arithmetic term first; if followed by CMP -> comparison.
        start = self.pos
        term = self._arith()
        if self._peek() in CMP:
            op = self._eat(); rhs = self._arith()
            return self._cmp(op, term, rhs)
        # not a comparison: term must itself be a Bool predicate
        if term is None:
            raise ValueError("empty atom")
        return term  # _arith returns Bool pred when it was a bare predicate

    def _cmp(self, op, a, b):
        if op in ('=',): return a == b
        if op in ('≠','!='): return a != b
        if op in ('>',): return a > b
        if op in ('<',): return a < b
        if op in ('≥','>='): return a >= b
        if op in ('≤','<='): return a <= b
        raise ValueError(f"bad cmp {op}")

    def _arith(self):
        left = self._arith_term()
        while self._peek() in ('+','-'):
            op=self._eat(); right=self._arith_term()
            left = (left+right) if op=='+' else (left-right)
        return left
    def _arith_term(self):
        left = self._factor()
        while self._peek() in ('*','/'):
            op=self._eat(); right=self._factor()
            left = (left*right) if op=='*' else (left/right)
        return left
    def _factor(self):
        tok = self._peek()
        if tok == '(':
            self._eat('('); e=self._iff(); self._eat(')'); return e
        if re.match(r'^\d', tok):
            self._eat()
            return IntVal(int(float(tok)))
        if tok.startswith("'") and tok.endswith("'"):
            self._eat()
            cname = tok.strip("'")
            if cname not in self.const_map:
                self.const_map[cname] = IntVal(len(self.const_map)+1)
            return self.const_map[cname]
        # name: predicate (Bool) or numeric function (Int) depending on following CMP/arith
        name = self._eat()
        args = []
        is_call = False
        if self._peek() == '(':
            is_call=True; self._eat('(')
            if self._peek() != ')':
                args.append(self._arith())
                while self._peek()==',':
                    self._eat(','); args.append(self._arith())
            self._eat(')')
        # Decide Bool vs Int by lookahead: if next is CMP/arith-op, this name is Int-valued
        nxt = self._peek()
        numeric_ctx = nxt in CMP or nxt in ('+','-','*','/')
        if name in self.scope: return self.scope[name]
        if not is_call and re.match(r'^[a-z][a-z0-9_]*$', name) and not args:
            # bare lowercase -> free var (numeric or term)
            if name not in self.free: self.free[name]=Int(name)
            return self.free[name]
        key=f"{name}_{len(args)}"
        if numeric_ctx:
            if key not in self.int_cache:
                sorts=[IntSort()]*len(args)+[IntSort()]
                self.int_cache[key]=Function(name+"_I", *sorts)
            return self.int_cache[key](*args) if args else self.int_cache[key]()
        else:
            if key not in self.func_cache:
                sorts=[IntSort()]*len(args)+[BoolSort()]
                self.func_cache[key]=Function(name, *sorts)
            return self.func_cache[key](*args) if args else self.func_cache[key]()

print("FOL parser v11 ready (handles forall/exists/->/<->/not/and/or + arithmetic + string-literals)")


FOL parser v11 ready (handles forall/exists/->/<->/not/and/or + arithmetic + string-literals)


In [10]:
# ==================================================================
# STAGE 4 -- v12 Pipeline
# FOL-string Pass-2 + BoN Self-Consistency + Predicate Grounding
# + Confidence Gating (3-tier) + idx Filter + Unsat-Core Explanation
# ==================================================================
import re, time, json
from dataclasses import dataclass, field
from collections import Counter
from z3 import Solver, Not, Bool, sat, unsat
from vllm import SamplingParams


@dataclass
class PipelineResult:
    sample_id: int
    status: str = "pending"
    z3_status: str = "pending"
    z3_compiled: int = 0
    z3_total: int = 0
    z3_attempts: int = 0
    z3_errors: list = field(default_factory=list)
    hallucination_warn: list = field(default_factory=list)
    local_ontology: list = field(default_factory=list)
    premises_ast: list = field(default_factory=list)
    question_fol: list = field(default_factory=list)
    predicted_answers: list = field(default_factory=list)
    ground_truth: list = field(default_factory=list)
    total_questions: int = 0
    correct_count: int = 0
    time_sec: float = 0.0
    error_log: list = field(default_factory=list)
    answer_source: list = field(default_factory=list)


# ---------------- Question type & embedded-FOL detection ----------------
def is_mc_question(q: str) -> bool:
    return bool(re.search(r'\n\s*[ABCD][\.\)]', q))


def extract_embedded_fol(q: str):
    """If a Yes/No question already contains a FOL Statement, return the FOL string."""
    if not re.search(r'[\u2200\u2203\u2192\u00ac\u2227\u2228\u2194]', q):
        return None
    if is_mc_question(q):
        return None
    parts = re.split(r'[Ss]tatement\s*:', q)
    if len(parts) > 1:
        cand = parts[-1].strip().strip("'\"").strip()
        # strip trailing punctuation like '?' or '.' if at end
        cand = re.sub(r'[\?\.]\s*$', '', cand).strip()
        return cand or None
    return None


# ---------------- Predicate grounding (exact, case-insensitive only) ----------------
def extract_predicates_used(fol_str: str):
    # Predicates are written with name DIRECTLY adjacent to '(' (no space).
    # Variables in '∀x (' have a space before '(' and won't match.
    return set(re.findall(r'([A-Za-z_]\w*)\(', fol_str))


def ground_predicates(fol_str: str, allowed: set):
    """Map predicate names to canonical names in `allowed` via exact case-insensitive match.
    Returns (grounded_str, ungrounded_set). Empty ungrounded_set => fully grounded."""
    used = extract_predicates_used(fol_str)
    ungrounded = set()
    out = fol_str
    allowed_lower = {p.lower(): p for p in allowed}
    for p in used:
        if p in allowed:
            continue
        canon = allowed_lower.get(p.lower())
        if canon and canon != p:
            out = re.sub(rf'\b{re.escape(p)}\b', canon, out)
        elif not canon:
            ungrounded.add(p)
    return out, ungrounded


# ---------------- Pass-2 prompts (FOL-string output) ----------------
PASS2_YN_SYSTEM = (
    "You translate Yes/No questions into First-Order Logic. "
    "Use EXACTLY the same syntax and predicate names as the premises shown below. "
    "Allowed symbols ONLY: \u2200 \u2203 \u2192 \u00ac \u2227 \u2228 \u2194. "
    "Reuse predicate names verbatim from the premises. "
    "Output ONLY the formula on a single line. No quotes, no labels, no explanation."
)

PASS2_MC_SYSTEM = (
    "You translate each of the 4 multiple-choice options into First-Order Logic. "
    "Use EXACTLY the same syntax and predicate names as the premises shown below. "
    "Allowed symbols ONLY: \u2200 \u2203 \u2192 \u00ac \u2227 \u2228 \u2194. "
    "Reuse predicate names verbatim. Output EXACTLY 4 lines:\n"
    "A: <formula>\nB: <formula>\nC: <formula>\nD: <formula>\n"
    "No explanation."
)


def _relevant_premise_fols(sample, q_idx):
    fols = sample.get('premises-FOL', [])
    idx_list = sample.get('idx', [])
    if USE_IDX_FILTER and q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
        rel = [fols[j-1] for j in idx_list[q_idx] if 0 <= j-1 < len(fols)]
        if rel:
            return rel
    return fols


def _ontology_str(preds: set, max_chars=600):
    return ", ".join(sorted(preds))[:max_chars]


def make_pass2_yn_user(sample, q, q_idx, preds):
    rel = _relevant_premise_fols(sample, q_idx)
    shots = "\n".join(f"- {f}" for f in rel)
    onto = _ontology_str(preds)
    return (f"Relevant premises (FOL syntax to mirror):\n{shots}\n\n"
            f"Available predicate names: {onto}\n\n"
            f"Question: {q}\n\nFormula:")


def make_pass2_mc_user(sample, q, q_idx, preds):
    rel = _relevant_premise_fols(sample, q_idx)
    shots = "\n".join(f"- {f}" for f in rel)
    onto = _ontology_str(preds)
    return (f"Relevant premises (FOL syntax to mirror):\n{shots}\n\n"
            f"Available predicate names: {onto}\n\n"
            f"Question (multiple choice):\n{q}\n\n"
            f"Write each option as a FOL formula:")


# ---------------- Parse Pass-2 outputs ----------------
def parse_yn_output(raw: str) -> str:
    """Extract a single FOL formula from a (possibly noisy) Yes/No Pass-2 output."""
    raw = (raw or "").strip()
    for line in reversed(raw.split('\n')):
        line = line.strip().strip('`').strip()
        line = re.sub(r'^(?:formula|fol|answer|statement)\s*[:=]\s*', '', line, flags=re.I)
        if re.search(r'[\u2200\u2203\u2192\u00ac\u2227\u2228\u2194]', line) or re.search(r'[A-Za-z_]\w*\s*\(', line):
            return line
    # last resort: last non-empty line
    lines = [l for l in raw.split('\n') if l.strip()]
    return lines[-1].strip() if lines else ""


def parse_mc_output(raw: str) -> dict:
    """Extract a dict {A,B,C,D} -> FOL string from a Pass-2 MC output."""
    out = {}
    for line in (raw or "").split('\n'):
        m = re.match(r'^\s*([ABCD])\s*[:\.\)]\s*(.+)$', line.strip())
        if m:
            f = m.group(2).strip().strip('`').strip()
            out[m.group(1)] = f
    return out


# ---------------- Premise FOL parsing ----------------
def parse_gold_premises(fol_strings):
    fc, cm, ic = {}, {}, {}
    exprs, per_idx = [], {}
    n_ok = 0
    for i, f in enumerate(fol_strings):
        try:
            e = FOLParser(fc, cm, ic).parse(f)
            exprs.append(e)
            per_idx[i] = e
            n_ok += 1
        except Exception:
            pass
    all_ok = (n_ok == len(fol_strings)) and len(fol_strings) > 0
    preds = set(k.rsplit('_', 1)[0] for k in fc.keys() if not k.startswith('__'))
    return dict(exprs=exprs, per_idx=per_idx, fc=fc, cm=cm, ic=ic,
                preds=preds, all_ok=all_ok, n_ok=n_ok, n_tot=len(fol_strings))


def _select_premises(g, sample, q_idx):
    if not USE_IDX_FILTER:
        return g['exprs']
    idx_list = sample.get('idx', [])
    if q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
        rel = [g['per_idx'][j-1] for j in idx_list[q_idx] if (j-1) in g['per_idx']]
        if rel:
            return rel
    return g['exprs']


# ---------------- Entailment ----------------
def _solver_with(exprs):
    s = Solver(); s.set('timeout', Z3_SOLVER_TIMEOUT)
    for e in exprs:
        s.add(e)
    return s


def entail_yn(premise_exprs, stmt):
    s = _solver_with(premise_exprs)
    s.push(); s.add(Not(stmt))
    try:
        r1 = s.check()
    except Exception:
        r1 = None
    s.pop()
    if r1 == unsat:
        return 'Yes'
    s.push(); s.add(stmt)
    try:
        r2 = s.check()
    except Exception:
        r2 = None
    s.pop()
    if r2 == unsat:
        return 'No'
    return 'Unknown'


def entail_mc(premise_exprs, options_exprs):
    if not options_exprs:
        return 'Unknown'
    entailed, consistent, contradicted = [], [], []
    for lab, oe in options_exprs.items():
        s = _solver_with(premise_exprs)
        s.push(); s.add(Not(oe))
        try: r1 = s.check()
        except Exception: r1 = None
        s.pop()
        if r1 == unsat:
            entailed.append(lab)
        s.push(); s.add(oe)
        try: r2 = s.check()
        except Exception: r2 = None
        s.pop()
        if r2 == unsat:
            contradicted.append(lab)
        elif r2 == sat:
            consistent.append(lab)
    if len(entailed) == 1:
        return entailed[0]
    if len(entailed) > 1:
        return entailed[0]
    nonc = [l for l in options_exprs if l in consistent and l not in contradicted]
    if len(nonc) == 1:
        return nonc[0]
    if len(consistent) == 1:
        return consistent[0]
    return 'Unknown'


# ---------------- Unsat-core explanation (XAI deliverable) ----------------
def unsat_core_premises(premise_exprs_with_idx, stmt):
    """premise_exprs_with_idx: list of (premise_z3_expr, original_index).
    Returns list of original indices whose premises were used in the proof, or []."""
    s = Solver(); s.set('timeout', Z3_SOLVER_TIMEOUT)
    bools = []
    for k, (e, orig_i) in enumerate(premise_exprs_with_idx):
        b = Bool(f'__core_p{k}')
        s.assert_and_track(e, b)
        bools.append((b, orig_i))
    s.add(Not(stmt))
    try:
        if s.check() == unsat:
            core_strs = set(str(c) for c in s.unsat_core())
            return [orig_i for (b, orig_i) in bools if str(b) in core_strs]
    except Exception:
        pass
    return []


# ---------------- Self-consistency vote ----------------
def sc_vote_yn(premise_exprs, candidates, preds, fc, cm, ic):
    """Vote over Yes/No/Unknown from BoN FOL-string candidates."""
    votes = Counter()
    n_parsed = 0
    grounded_fols = []
    for cand in candidates:
        if not cand:
            votes['_empty'] += 1
            continue
        grounded, ung = ground_predicates(cand, preds)
        if ung:
            votes['_ungrounded'] += 1
            continue
        try:
            stmt = FOLParser(fc, cm, ic).parse(grounded)
        except Exception:
            votes['_parse_err'] += 1
            continue
        n_parsed += 1
        grounded_fols.append((grounded, stmt))
        votes[entail_yn(premise_exprs, stmt)] += 1
    return votes, n_parsed, grounded_fols


def sc_vote_mc(premise_exprs, candidate_sets, preds, fc, cm, ic):
    """Vote over A/B/C/D/Unknown from BoN candidate sets."""
    votes = Counter()
    n_parsed = 0
    for cand in candidate_sets:
        if not isinstance(cand, dict) or len(cand) < 2:
            votes['_empty'] += 1
            continue
        opt_exprs = {}
        ok = True
        for lab in 'ABCD':
            if lab not in cand:
                continue
            g, ung = ground_predicates(cand[lab], preds)
            if ung:
                ok = False; break
            try:
                opt_exprs[lab] = FOLParser(fc, cm, ic).parse(g)
            except Exception:
                ok = False; break
        if not ok or len(opt_exprs) < 2:
            votes['_failed'] += 1
            continue
        n_parsed += 1
        votes[entail_mc(premise_exprs, opt_exprs)] += 1
    return votes, n_parsed


def gate_decision(votes: Counter, total_candidates: int):
    """3-tier gate: returns (best_answer, confidence, tier).
    tier in {'z3_trust', 'z3_arbiter', 'fallback'}."""
    real = {k: v for k, v in votes.items() if not k.startswith('_')}
    if not real:
        return None, 0.0, 'fallback'
    total = sum(real.values())
    ans, c = max(real.items(), key=lambda x: x[1])
    conf = c / total
    grounded_frac = total / max(total_candidates, 1)
    definite = ans not in (None, 'Unknown')
    if definite and conf >= SC_CONFIDENCE_TAU and grounded_frac >= GROUNDED_FRAC_TAU:
        return ans, conf, 'z3_trust'
    if definite and conf >= 0.5:
        return ans, conf, 'z3_arbiter'
    return ans, conf, 'fallback'


# ---------------- BoN generation helper ----------------
def batch_generate_n(prompt_pairs, max_tokens, n=5, temperature=0.7, enable_thinking=False):
    """Generate N samples per prompt using vLLM n parameter. Returns list[list[str]]."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=enable_thinking)
        except TypeError:
            text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        formatted.append(text)
    params = SamplingParams(
        temperature=temperature, top_p=0.95, max_tokens=max_tokens, n=n,
        repetition_penalty=1.1,
    )
    outputs = llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [[o.text.strip() for o in out.outputs] for out in outputs_sorted]


# ---------------- Answer-fallback prompt (reuse from utils) ----------------
def _make_answer_user(sample, question):
    lines = ["### Premises"]
    for i, p in enumerate(sample.get('premises-NL', [])):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"


def _make_informed_answer_user(sample, question, z3_hint, z3_conf, votes_summary, has_z3):
    """v12.1 -- include premise FOL and Z3 evidence so Qwen can reason informedly."""
    nls = sample.get('premises-NL', [])
    fols = sample.get('premises-FOL', [])
    lines = ["### Premises (NL + FOL)"]
    for i, p in enumerate(nls):
        lines.append(f"P{i+1}. {p}")
        if i < len(fols) and fols[i]:
            lines.append(f"      FOL: {fols[i]}")
    out = "\n".join(lines) + f"\n\n### Question\n{question}"
    if has_z3 and z3_hint:
        out += "\n\n### Symbolic evidence (Z3 evaluated 5 formalizations)"
        out += f"\n- Vote distribution: {votes_summary or '(empty)'}"
        out += f"\n- Z3's tentative answer: {z3_hint} (confidence {z3_conf:.2f})"
        out += ("\n- Note: Z3 may over-claim entailment on statements that are "
                "converses/inverses of premises or have unscoped free variables. "
                "Verify carefully by reasoning step-by-step.")
    return out



# ============================== MAIN ==============================
def run_batch_pipeline(samples, dataset_name="Dataset"):
    N = len(samples)
    t0 = time.time()
    results = [PipelineResult(sample_id=i,
                              ground_truth=s.get('answers', []),
                              total_questions=len(s.get('questions', [])))
               for i, s in enumerate(samples)]

    # ---- Stage A: parse premises ----
    print("\n" + "=" * 65)
    print("  STAGE A -- Premise FOL parse (gold)")
    print("=" * 65)
    gold = {}
    for i, s in enumerate(samples):
        g = parse_gold_premises(s.get('premises-FOL', []))
        gold[i] = g
        results[i].z3_compiled = g['n_ok']
        results[i].z3_total = g['n_tot']
        results[i].z3_attempts = 1
        results[i].local_ontology = [{"predicate": k.rsplit('_', 1)[0],
                                      "arity": int(k.rsplit('_', 1)[1])}
                                     for k in g['fc'].keys() if not k.startswith('__')]
        if g['exprs'] and (g['all_ok'] or g['n_ok'] >= max(1, g['n_tot'] - 1)):
            results[i].z3_status = 'sat'
        else:
            results[i].z3_status = 'no_ast'
    print("  Premise status:", dict(Counter(r.z3_status for r in results)))

    # ---- Stage B: dispatch questions by type ----
    print("\n" + "=" * 65)
    print("  STAGE B -- Question dispatch")
    print("=" * 65)
    embedded = []        # (i, q_idx, fol_str)
    yn_tasks = []        # (i, q_idx, q_str)
    mc_tasks = []        # (i, q_idx, q_str)
    skip_to_qwen = []    # premise parse failed -> direct Qwen

    for i, s in enumerate(samples):
        if results[i].z3_status != 'sat':
            for q_idx, q in enumerate(s.get('questions', [])):
                skip_to_qwen.append((i, q_idx, q, None, 0.0, 'no_premise', ''))
            continue
        for q_idx, q in enumerate(s.get('questions', [])):
            emb = extract_embedded_fol(q)
            if emb:
                embedded.append((i, q_idx, emb))
            elif is_mc_question(q):
                mc_tasks.append((i, q_idx, q))
            else:
                yn_tasks.append((i, q_idx, q))
    print(f"  embedded-FOL: {len(embedded)} | Pass-2 YN: {len(yn_tasks)} | "
          f"Pass-2 MC: {len(mc_tasks)} | no-premise: {len(skip_to_qwen)}")

    # ---- Stage C: Pass-2 generation (BoN) ----
    print("\n" + "=" * 65)
    print(f"  STAGE C -- Pass-2 BoN (N={BON_N_QFORMALIZE})")
    print("=" * 65)
    yn_cands = {}
    if yn_tasks:
        prompts = [(PASS2_YN_SYSTEM,
                    make_pass2_yn_user(samples[i], q, q_idx, gold[i]['preds']))
                   for (i, q_idx, q) in yn_tasks]
        gen = batch_generate_n(prompts, MAX_PASS2_TOKENS_YN, n=BON_N_QFORMALIZE,
                               temperature=BON_TEMP, enable_thinking=False)
        for k, cands in enumerate(gen):
            i, q_idx, _ = yn_tasks[k]
            yn_cands[(i, q_idx)] = [parse_yn_output(c) for c in cands]
        print(f"  YN BoN done: {len(yn_tasks)} questions x {BON_N_QFORMALIZE} cands")

    mc_cands = {}
    if mc_tasks:
        prompts = [(PASS2_MC_SYSTEM,
                    make_pass2_mc_user(samples[i], q, q_idx, gold[i]['preds']))
                   for (i, q_idx, q) in mc_tasks]
        gen = batch_generate_n(prompts, MAX_PASS2_TOKENS_MC, n=BON_N_QFORMALIZE,
                               temperature=BON_TEMP, enable_thinking=False)
        for k, cands in enumerate(gen):
            i, q_idx, _ = mc_tasks[k]
            mc_cands[(i, q_idx)] = [parse_mc_output(c) for c in cands]
        print(f"  MC BoN done: {len(mc_tasks)} questions x {BON_N_QFORMALIZE} cands")

    # ---- Stage D: entailment, gating, embedded direct ----
    print("\n" + "=" * 65)
    print("  STAGE D -- Entailment + Gating")
    print("=" * 65)
    stats = Counter()
    qwen_queue = []   # (i, q_idx, q, z3_hint, z3_conf, reason)

    # D.1 embedded direct
    for (i, q_idx, fol) in embedded:
        g = gold[i]
        prem = _select_premises(g, samples[i], q_idx)
        # build (expr, orig_idx) for unsat-core
        idx_list = samples[i].get('idx', [])
        if USE_IDX_FILTER and q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
            prem_with_idx = [(g['per_idx'][j-1], j-1) for j in idx_list[q_idx] if (j-1) in g['per_idx']]
            if not prem_with_idx:
                prem_with_idx = [(g['per_idx'][k], k) for k in g['per_idx']]
        else:
            prem_with_idx = [(g['per_idx'][k], k) for k in g['per_idx']]

        g_str, ung = ground_predicates(fol, g['preds'])
        if ung:
            qwen_queue.append((i, q_idx, samples[i]['questions'][q_idx], None, 0.0, 'embed_ungrounded', ''))
            stats['embed_ungrounded'] += 1
            continue
        try:
            stmt = FOLParser(g['fc'], g['cm'], g['ic']).parse(g_str)
        except Exception as e:
            qwen_queue.append((i, q_idx, samples[i]['questions'][q_idx], None, 0.0, 'embed_parse_err', ''))
            stats['embed_parse_err'] += 1
            results[i].error_log.append(f"q{q_idx} embed parse: {str(e)[:80]}")
            continue
        ans = entail_yn(prem, stmt)
        core_ids = []
        if EXTRACT_UNSAT_CORE and ans == 'Yes':
            core_ids = unsat_core_premises(prem_with_idx, stmt)
        results[i].predicted_answers.append({
            'question_id': q_idx, 'answer': ans,
            'reasoning': f'[Z3-direct] embedded FOL, conf=1.0',
            'support_idx': core_ids,
        })
        results[i].answer_source.append('z3_direct')
        stats['z3_direct'] += 1

    # D.2 YN BoN+SC
    for (i, q_idx, q) in yn_tasks:
        g = gold[i]
        prem = _select_premises(g, samples[i], q_idx)
        cands = yn_cands.get((i, q_idx), [])
        votes, n_parsed, grounded_fols = sc_vote_yn(prem, cands, g['preds'],
                                                    g['fc'], g['cm'], g['ic'])
        ans, conf, tier = gate_decision(votes, len(cands))
        results[i].question_fol.append({'q_id': q_idx, 'votes': dict(votes),
                                        'parsed': n_parsed})
        if tier == 'z3_trust':
            core_ids = []
            if EXTRACT_UNSAT_CORE and ans == 'Yes' and grounded_fols:
                idx_list = samples[i].get('idx', [])
                if USE_IDX_FILTER and q_idx < len(idx_list) and isinstance(idx_list[q_idx], list) and idx_list[q_idx]:
                    pwi = [(g['per_idx'][j-1], j-1) for j in idx_list[q_idx] if (j-1) in g['per_idx']]
                else:
                    pwi = [(g['per_idx'][k], k) for k in g['per_idx']]
                if pwi:
                    core_ids = unsat_core_premises(pwi, grounded_fols[0][1])
            results[i].predicted_answers.append({
                'question_id': q_idx, 'answer': ans,
                'reasoning': f'[Z3-SC] tier1 conf={conf:.2f} votes={dict(votes)}',
                'support_idx': core_ids,
            })
            results[i].answer_source.append('z3_trust')
            stats['z3_trust'] += 1
        else:
            vs = ', '.join(f'{k}={v}' for k, v in votes.items() if not k.startswith('_'))
            qwen_queue.append((i, q_idx, q, ans, conf, tier, vs))
            stats[f'queue_{tier}'] += 1

    # D.3 MC BoN+SC
    for (i, q_idx, q) in mc_tasks:
        g = gold[i]
        prem = _select_premises(g, samples[i], q_idx)
        cands = mc_cands.get((i, q_idx), [])
        votes, n_parsed = sc_vote_mc(prem, cands, g['preds'], g['fc'], g['cm'], g['ic'])
        ans, conf, tier = gate_decision(votes, len(cands))
        if tier == 'z3_trust':
            results[i].predicted_answers.append({
                'question_id': q_idx, 'answer': ans,
                'reasoning': f'[Z3-SC-MC] tier1 conf={conf:.2f} votes={dict(votes)}',
                'support_idx': [],
            })
            results[i].answer_source.append('z3_trust')
            stats['z3_trust_mc'] += 1
        else:
            vs = ', '.join(f'{k}={v}' for k, v in votes.items() if not k.startswith('_'))
            qwen_queue.append((i, q_idx, q, ans, conf, tier, vs))
            stats[f'queue_mc_{tier}'] += 1

    print(f"  Gating stats: {dict(stats)}")
    print(f"  Qwen queue: {len(qwen_queue)} questions")

    # ---- Stage E (v12.1): Z3-informed Qwen CoT + Qwen-priority arbiter ----
    full_queue = list(qwen_queue) + skip_to_qwen
    if full_queue:
        print("\n" + "=" * 65)
        print(f"  STAGE E -- Z3-informed Qwen CoT ({len(full_queue)} questions)")
        print("=" * 65)
        prompts = []
        for (i, q_idx, q, z3_hint, z3_conf, reason, votes_summary) in full_queue:
            if USE_Z3_INFORMED_FALLBACK:
                user = _make_informed_answer_user(samples[i], q, z3_hint, z3_conf,
                                                  votes_summary, has_z3=bool(z3_hint))
            else:
                user = _make_answer_user(samples[i], q)
            prompts.append((ANSWER_SYSTEM, user))
        responses = batch_generate(prompts, ANS_MAX_TOKENS, enable_thinking=True, use_lora=False)
        for k, raw in enumerate(responses):
            i, q_idx, q, z3_hint, z3_conf, reason, votes_summary = full_queue[k]
            qwen_ans = extract_final_answer(raw)
            if qwen_ans == 'UNPARSEABLE':
                qwen_ans = safe_json(raw).get('answer', 'Unknown')

            # v12.1 ARBITER: Z3 only wins on AGREEMENT.
            # On disagreement, Qwen wins -- diagnostic shows high-conf Z3 disagreement
            # is the signature of 'No'-class semantic gap where Z3 over-claims.
            if z3_hint and z3_conf >= 0.5 and reason == 'z3_arbiter':
                qwen_norm = str(qwen_ans).strip().upper()
                z3_norm = str(z3_hint).strip().upper()
                if qwen_norm == z3_norm:
                    final = z3_hint
                    src = 'z3_arbiter_agree'
                    info = f'[Z3+Qwen agree] z3_conf={z3_conf:.2f}'
                else:
                    # FLIP from v12: on disagreement, always trust Qwen
                    final = qwen_ans
                    src = 'qwen_arbiter'
                    info = f'[Qwen wins (v12.1 flip)] z3={z3_hint}@{z3_conf:.2f}'
            else:
                final = qwen_ans
                src = 'qwen_fallback'
                info = f'[Qwen fallback{" + Z3 hint" if USE_Z3_INFORMED_FALLBACK and z3_hint else ""}] reason={reason}'

            results[i].predicted_answers.append({
                'question_id': q_idx, 'answer': final,
                'reasoning': info, 'support_idx': [],
            })
            results[i].answer_source.append(src)

    # ---- Scoring ----
    for i, r in enumerate(results):
        # Sort by q_id to align with ground truth order
        order = sorted(range(len(r.predicted_answers)),
                       key=lambda k: r.predicted_answers[k]['question_id'])
        r.predicted_answers = [r.predicted_answers[k] for k in order]
        r.answer_source = [r.answer_source[k] for k in order]
        gt = r.ground_truth
        r.correct_count = sum(
            1 for a in r.predicted_answers
            if a['question_id'] < len(gt)
            and str(a['answer']).strip().upper() == str(gt[a['question_id']]).strip().upper()
        )
        r.status = 'success' if r.z3_status == 'sat' else 'failed'
        r.time_sec = round(time.time() - t0, 2)

    print(f"\n  Total wall-time: {time.time()-t0:.1f}s")
    return results


print("Pipeline v12.1 (Z3-informed fallback + Qwen-priority arbiter) loaded")


Pipeline v12.1 (Z3-informed fallback + Qwen-priority arbiter) loaded


In [11]:
# ==================================================================
# STAGE 5 -- v12 Evaluation & Export
# v12 sources: z3_direct | z3_trust | z3_arbiter_agree | z3_arbiter_override
#              | qwen_arbiter | qwen_fallback
# ==================================================================
from pathlib import Path

V12_Z3_SOURCES = {'z3_direct', 'z3_trust', 'z3_arbiter_agree', 'z3_arbiter_override'}
V12_QWEN_SOURCES = {'qwen_arbiter', 'qwen_fallback'}


def evaluate(results):
    n = len(results)
    if n == 0:
        return {}
    total_q = sum(r.total_questions for r in results)
    total_ok = sum(r.correct_count for r in results)
    status_ct = Counter()
    z3_ct = Counter()
    for r in results:
        status_ct[r.status] += 1
        z3_ct[r.z3_status] += 1

    # Detailed source breakdown
    src_ct = Counter()
    src_correct = Counter()
    for r in results:
        gt = r.ground_truth
        for j, ar in enumerate(r.predicted_answers):
            qid = ar['question_id']
            if qid >= len(gt):
                continue
            src = r.answer_source[j] if j < len(r.answer_source) else 'unknown'
            src_ct[src] += 1
            if str(ar['answer']).strip().upper() == str(gt[qid]).strip().upper():
                src_correct[src] += 1

    # Bucket Z3 vs Qwen
    z3_correct = sum(src_correct[s] for s in V12_Z3_SOURCES)
    z3_total = sum(src_ct[s] for s in V12_Z3_SOURCES)
    qw_correct = sum(src_correct[s] for s in V12_QWEN_SOURCES)
    qw_total = sum(src_ct[s] for s in V12_QWEN_SOURCES)

    # Per-source accuracy
    per_src_acc = {s: (src_correct[s], src_ct[s],
                       round(src_correct[s]/src_ct[s], 4) if src_ct[s] else 0.0)
                   for s in (V12_Z3_SOURCES | V12_QWEN_SOURCES) if src_ct[s] > 0}

    return {
        'n_samples': n,
        'total_questions': total_q,
        'total_correct': total_ok,
        'accuracy': round(total_ok / total_q, 4) if total_q else 0,
        'status_breakdown': dict(status_ct),
        'z3_breakdown': dict(z3_ct),
        'source_counts': dict(src_ct),
        'source_correct': dict(src_correct),
        'per_source_accuracy': per_src_acc,
        'z3_bucket_accuracy': round(z3_correct/z3_total, 4) if z3_total else 0,
        'z3_bucket_correct': z3_correct, 'z3_bucket_total': z3_total,
        'qwen_bucket_accuracy': round(qw_correct/qw_total, 4) if qw_total else 0,
        'qwen_bucket_correct': qw_correct, 'qwen_bucket_total': qw_total,
    }


def result_to_dict(r):
    return {
        'sample_id': r.sample_id, 'status': r.status,
        'z3_status': r.z3_status, 'z3_compiled': r.z3_compiled, 'z3_total': r.z3_total,
        'local_ontology_size': len(r.local_ontology),
        'predicted': [a['answer'] for a in r.predicted_answers],
        'answer_sources': r.answer_source,
        'ground_truth': r.ground_truth,
        'per_question': [
            {'q_id': a['question_id'], 'pred': a['answer'],
             'gt': r.ground_truth[a['question_id']] if a['question_id'] < len(r.ground_truth) else '?',
             'correct': (str(a['answer']).upper() == str(r.ground_truth[a['question_id']]).upper()
                         if a['question_id'] < len(r.ground_truth) else False),
             'source': r.answer_source[j] if j < len(r.answer_source) else '?',
             'support_idx': a.get('support_idx', []),
             'reasoning': a.get('reasoning', '')[:200]}
            for j, a in enumerate(r.predicted_answers)
        ],
        'time_sec': r.time_sec,
        'error_log': r.error_log[-2:],
    }


def finalize_and_save(results, output_path, dataset_path, dataset_name='Dataset'):
    if not results:
        print(f"[WARN] No results for {dataset_name}")
        return
    metrics = evaluate(results)
    acc = metrics.get('accuracy', 0)

    W = 70
    print('\n' + '=' * W)
    print(f"  {dataset_name.upper()} -- v12.1 EVALUATION SUMMARY")
    print('=' * W)
    print(f"  Model       : {QWEN_MODEL_ID}")
    print(f"  Samples     : {metrics['n_samples']}")
    print(f"  Questions   : {metrics['total_questions']}")
    print(f"  Correct     : {metrics['total_correct']}")
    print(f"  ACCURACY    : {acc:.2%}")
    print('-' * W)
    print('  Premise parse status:')
    for k, v in metrics['z3_breakdown'].items():
        if v: print(f"    {k:20s}: {v:3d}")
    print('-' * W)
    print('  Answer-source accuracy:')
    for src, (c, t, a) in sorted(metrics['per_source_accuracy'].items(),
                                  key=lambda x: -x[1][1]):
        bar = '#' * min(int(a * 30), 30)
        print(f"    {src:25s}: {c:3d}/{t:3d} = {a:.1%}  {bar}")
    print('-' * W)
    print(f"  Z3 bucket  : {metrics['z3_bucket_correct']}/{metrics['z3_bucket_total']} = {metrics['z3_bucket_accuracy']:.1%}")
    print(f"  Qwen bucket: {metrics['qwen_bucket_correct']}/{metrics['qwen_bucket_total']} = {metrics['qwen_bucket_accuracy']:.1%}")
    print('=' * W)

    # Per-sample (compact)
    hdr = f"{'ID':>3} | {'Status':>8} | {'Z3':>10} | {'Corr':>5} | first-srcs"
    print(hdr); print('-' * len(hdr))
    for r in results[:30]:
        srcs = '/'.join(s[:6] for s in r.answer_source[:2]) if r.answer_source else '?'
        print(f"{r.sample_id:>3} | {r.status:>8} | {r.z3_status:>10} | "
              f"{r.correct_count}/{r.total_questions:>3} | {srcs}")
    if len(results) > 30:
        print(f"  ... ({len(results) - 30} more)")

    output_data = {
        'meta': {
            'model': QWEN_MODEL_ID, 'engine': 'vLLM',
            'tensor_parallel': TENSOR_PARALLEL,
            'pipeline_version': 'v12_1_z3_informed_fallback',
            'n_samples': len(results),
            'config': {
                'PREMISE_SOURCE': PREMISE_SOURCE,
                'USE_IDX_FILTER': USE_IDX_FILTER,
                'BON_N_QFORMALIZE': BON_N_QFORMALIZE,
                'BON_TEMP': BON_TEMP,
                'SC_CONFIDENCE_TAU': SC_CONFIDENCE_TAU,
                'GROUNDED_FRAC_TAU': GROUNDED_FRAC_TAU,
                'EXTRACT_UNSAT_CORE': EXTRACT_UNSAT_CORE,
                'USE_Z3_INFORMED_FALLBACK': USE_Z3_INFORMED_FALLBACK,
            },
            'dataset': dataset_path,
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        },
        'metrics': metrics,
        'per_sample': [result_to_dict(r) for r in results],
    }
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    fsize = Path(output_path).stat().st_size / 1024
    print(f"\nSaved: {output_path} ({fsize:.1f} KB)")
    print(f"Final accuracy: {acc:.2%}  ({metrics['total_correct']}/{metrics['total_questions']})")


# ==================================================================
# RUN
# ==================================================================
print('\n' + '=' * 70)
print('  NEURO-SYMBOLIC PIPELINE v12.1 (Z3-informed fallback + Qwen-priority arbiter)')
print(f"  Model: {QWEN_MODEL_ID}  |  TP: {TENSOR_PARALLEL}")
print(f"  PREMISE_SOURCE={PREMISE_SOURCE}  USE_IDX_FILTER={USE_IDX_FILTER}  "
      f"BoN={BON_N_QFORMALIZE} TAU={SC_CONFIDENCE_TAU}")
print('=' * 70)

logic_samples = load_dataset(DATASET_PATH, is_physics=False)
print(f"\nLogic Dataset: {len(logic_samples)} samples")
if logic_samples:
    logic_results = run_batch_pipeline(logic_samples, dataset_name='Logic Dataset')
    finalize_and_save(logic_results, OUTPUT_PATH, DATASET_PATH, 'Logic Dataset')

print('\n' + '=' * 70)
print('  PIPELINE v12.1 HOAN TAT')
print('=' * 70)



  NEURO-SYMBOLIC PIPELINE v12.1 (Z3-informed fallback + Qwen-priority arbiter)
  Model: /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1  |  TP: 2
  PREMISE_SOURCE=gold  USE_IDX_FILTER=True  BoN=5 TAU=0.6

Logic Dataset: 411 samples

  STAGE A -- Premise FOL parse (gold)
  Premise status: {'sat': 407, 'no_ast': 4}

  STAGE B -- Question dispatch
  embedded-FOL: 48 | Pass-2 YN: 380 | Pass-2 MC: 376 | no-premise: 8

  STAGE C -- Pass-2 BoN (N=5)


Rendering prompts:   0%|          | 0/380 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1900 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(Worker_TP0 pid=248) WARNING 05-31 13:15:13 [jit_monitor.py:103] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(Worker_TP0 pid=248) WARNING 05-31 13:15:18 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts:  97%|█████████▋| 1850/1900 [01:05<00:00, 62.34it/s, est. speed input: 5444.00 toks/s, output: 622.11 toks/s]

...

Processed prompts: 100%|██████████| 1880/1880 [04:36<00:00,  6.80it/s, est. speed input: 1959.77 toks/s, output: 669.99 toks/s]


  MC BoN done: 376 questions x 5 cands

  STAGE D -- Entailment + Gating
  Gating stats: {'z3_direct': 46, 'embed_parse_err': 2, 'z3_trust': 229, 'queue_fallback': 145, 'queue_z3_arbiter': 6, 'z3_trust_mc': 262, 'queue_mc_fallback': 104, 'queue_mc_z3_arbiter': 10}
  Qwen queue: 267 questions

  STAGE E -- Z3-informed Qwen CoT (275 questions)


Rendering prompts:   0%|          | 0/275 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 275/275 [09:33<00:00,  2.08s/it, est. speed input: 318.72 toks/s, output: 287.17 toks/s]



  Total wall-time: 953.5s

  LOGIC DATASET -- v12.1 EVALUATION SUMMARY
  Model       : /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
  Samples     : 411
  Questions   : 812
  Correct     : 399
  ACCURACY    : 49.14%
----------------------------------------------------------------------
  Premise parse status:
    sat                 : 407
    no_ast              :   4
----------------------------------------------------------------------
  Answer-source accuracy:
    z3_trust                 : 294/491 = 59.9%  #################
    qwen_fallback            :  73/259 = 28.2%  ########
    z3_direct                :  27/ 46 = 58.7%  #################
    qwen_arbiter             :   1/ 11 = 9.1%  ##
    z3_arbiter_agree         :   4/  5 = 80.0%  ########################
----------------------------------------------------------------------
  Z3 bucket  : 325/542 = 60.0%
  Qwen bucket: 74/270 = 27.4%
 ID |   Status |         Z3 |  Corr | first-srcs
------------------------------